<a href="https://colab.research.google.com/github/AureleKarega/ML_Formative_3_GRP_3/blob/main/EM_GMM_Presentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


#Part 1: EM Algorithm with Gaussian Mixture Model (GMM)
## Step-by-step Presentation Notebook


## 1. Problem Idea

We want to model mixed height data (parents + children) using a **Gaussian Mixture Model**.

We assume:
- Data comes from 2 hidden distributions
- We don't know which point belongs to which group
- We use EM algorithm to learn it


## 2. Import Libraries

In [ ]:
import csv
import math
import pandas as pd

df = pd.read_csv("/content/GaltonFamilies.csv")
df.head()

,rownames,family,father,mother,midparentHeight,children,childNum,gender,childHeight
0,1,001,78.5,67.0,75.43,4,1,male,73.2
1,2,001,78.5,67.0,75.43,4,2,female,69.2
2,3,001,78.5,67.0,75.43,4,3,female,69.0
3,4,001,78.5,67.0,75.43,4,4,female,69.0
4,5,002,75.5,66.5,73.66,4,1,male,73.5



## 3. Load Data

We read CSV and extract:
- father height
- child height

We flatten them into one dataset because EM does not use labels.


In [ ]:

def load_data(filename):
    heights = []
    with open(filename, "r") as file:
        reader = csv.DictReader(file)
        for row in reader:
            heights.append(float(row["father"]))
            heights.append(float(row["childHeight"]))
    return heights



## 4. Gaussian Function

This computes probability of a value under a normal distribution:

Formula:
P(x) = (1 / sqrt(2πσ²)) * exp(-(x-μ)² / 2σ²)

- μ = mean
- σ² = variance


In [ ]:

def gaussian(x, mean, variance):
    return (1 / math.sqrt(2 * math.pi * variance)) * math.exp(-((x - mean) ** 2) / (2 * variance))



## 5. EM Algorithm (Core Idea)

We repeat two steps:

### E-step:
- Estimate probability each point belongs to each cluster

### M-step:
- Update:
  - mean
  - variance
  - mixing weights

We iterate until convergence.


In [ ]:

def EM(data, iterations=10):

    average = sum(data) / len(data)
    mean1 = average - 5
    mean2 = average + 5

    variance1 = 25
    variance2 = 25

    pi1 = 0.5
    pi2 = 0.5

    print("\nTracking Table")
    print("Iteration | mu1 | mu2 | var1 | var2 | pi1 | pi2 | LogLikelihood")

    for iteration in range(iterations):

        responsibilities1 = []
        responsibilities2 = []

        # E STEP
        for x in data:
            prob1 = pi1 * gaussian(x, mean1, variance1)
            prob2 = pi2 * gaussian(x, mean2, variance2)
            total = prob1 + prob2

            responsibilities1.append(prob1 / total)
            responsibilities2.append(prob2 / total)

        # M STEP
        N1 = sum(responsibilities1)
        N2 = sum(responsibilities2)

        mean1 = sum(responsibilities1[i] * data[i] for i in range(len(data))) / N1
        mean2 = sum(responsibilities2[i] * data[i] for i in range(len(data))) / N2

        variance1 = sum(responsibilities1[i] * (data[i] - mean1) ** 2 for i in range(len(data))) / N1
        variance2 = sum(responsibilities2[i] * (data[i] - mean2) ** 2 for i in range(len(data))) / N2

        pi1 = N1 / len(data)
        pi2 = N2 / len(data)

        log_likelihood = 0
        for x in data:
            value = pi1 * gaussian(x, mean1, variance1) + pi2 * gaussian(x, mean2, variance2)
            log_likelihood += math.log(value)

        print(iteration, round(mean1,2), round(mean2,2),
              round(variance1,2), round(variance2,2),
              round(pi1,2), round(pi2,2),
              round(log_likelihood,2))

    return mean1, mean2, variance1, variance2, pi1, pi2



## 6. Prediction Step

After training:
- We classify new height
- Compute probability under both Gaussians
- Choose higher probability


In [ ]:

def predict(height, model):

    mean1, mean2, var1, var2, pi1, pi2 = model

    if mean1 < mean2:
        child_mean, child_var, child_pi = mean1, var1, pi1
        parent_mean, parent_var, parent_pi = mean2, var2, pi2
    else:
        child_mean, child_var, child_pi = mean2, var2, pi2
        parent_mean, parent_var, parent_pi = mean1, var1, pi1

    child_probability = child_pi * gaussian(height, child_mean, child_var)
    parent_probability = parent_pi * gaussian(height, parent_mean, parent_var)

    total = child_probability + parent_probability

    print("\nPrediction for:", height)
    print("Child probability:", round(child_probability / total * 100,2), "%")
    print("Parent probability:", round(parent_probability / total * 100,2), "%")



## 7. Run the Model


In [ ]:

data = load_data("GaltonFamilies.csv")
model = EM(data)

test_height = float(input("Enter test height: "))
predict(test_height, model)



Tracking Table
Iteration | mu1 | mu2 | var1 | var2 | pi1 | pi2 | LogLikelihood
0 66.3 69.6 9.45 7.07 0.49 0.51 -4871.94
1 66.27 69.61 9.96 6.48 0.49 0.51 -4868.06
2 66.24 69.62 10.18 6.14 0.49 0.51 -4866.64
3 66.22 69.64 10.27 5.92 0.49 0.51 -4865.98
4 66.2 69.66 10.29 5.76 0.49 0.51 -4865.62
5 66.19 69.68 10.28 5.65 0.49 0.51 -4865.38
6 66.18 69.7 10.25 5.57 0.49 0.51 -4865.21
7 66.17 69.72 10.2 5.51 0.49 0.51 -4865.07
8 66.16 69.73 10.16 5.45 0.49 0.51 -4864.96
9 66.15 69.75 10.11 5.41 0.49 0.51 -4864.86
Enter test height: 67

Prediction for: 67.0
Child probability: 58.07 %
Parent probability: 41.93 %


# Part 2: Bayesian Probability — Sentiment Analysis
### IMDB Movie Reviews Dataset
---
**Objective:** Use Bayes' Theorem to compute the probability that a review is positive given a specific keyword.  
**Chosen direction:** P(Positive | keyword)  
**Dataset:** 50,000 IMDB movie reviews (25,000 positive, 25,000 negative)  
**Libraries used:** `csv`, `re` — no external ML libraries


## Step 1: Load the Dataset

We load the IMDB CSV file and store each review as a dictionary containing the review text (lowercased) and its sentiment label.


In [ ]:
import csv
import re

# ─── Load Data ───────────────────────────────────────────────
DATASET_PATH = "/content/IMDB Dataset.csv"

reviews = []
with open(DATASET_PATH, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        reviews.append({
            "text": row["review"].lower(),
            "sentiment": row["sentiment"].strip().lower()
        })

total      = len(reviews)
num_pos    = sum(1 for r in reviews if r["sentiment"] == "positive")
num_neg    = total - num_pos

print(f"Total reviews : {total:,}")
print(f"Positive      : {num_pos:,}  ({num_pos/total:.1%})")
print(f"Negative      : {num_neg:,}  ({num_neg/total:.1%})")


Total reviews : 50,000
Positive      : 25,000  (50.0%)
Negative      : 25,000  (50.0%)


## Step 2: Keyword Selection

We selected 4 keywords expected to indicate **positive** sentiment and 4 expected to indicate **negative** sentiment.

| Group    | Keywords |
|----------|----------|
| ✅ Positive | `excellent`, `masterpiece`, `brilliant`, `heartwarming` |
| ❌ Negative | `terrible`, `awful`, `boring`, `waste` |


In [ ]:
POSITIVE_KEYWORDS = ["excellent", "masterpiece", "brilliant", "heartwarming"]
NEGATIVE_KEYWORDS = ["terrible", "awful", "boring", "waste"]
ALL_KEYWORDS      = POSITIVE_KEYWORDS + NEGATIVE_KEYWORDS

print("Positive keywords:", POSITIVE_KEYWORDS)
print("Negative keywords:", NEGATIVE_KEYWORDS)


Positive keywords: ['excellent', 'masterpiece', 'brilliant', 'heartwarming']
Negative keywords: ['terrible', 'awful', 'boring', 'waste']


## Step 3: Keyword Search Helper

We use a regular expression with word boundaries (`\b`) to ensure we match whole words only — so "bore" won't match "boring", for example.


In [ ]:
def contains_keyword(text: str, keyword: str) -> bool:
    """Return True if keyword appears as a whole word in text."""
    return bool(re.search(r'\b' + re.escape(keyword) + r'\b', text))

def count_keyword(keyword: str):
    """Count how many positive and negative reviews contain the keyword."""
    in_pos = sum(1 for r in reviews
                 if r["sentiment"] == "positive" and contains_keyword(r["text"], keyword))
    in_neg = sum(1 for r in reviews
                 if r["sentiment"] == "negative" and contains_keyword(r["text"], keyword))
    return in_pos, in_neg, in_pos + in_neg

# Quick sanity check
in_pos, in_neg, total_kw = count_keyword("excellent")
print(f"'excellent' → in positive: {in_pos}, in negative: {in_neg}, total: {total_kw}")


'excellent' → in positive: 2868, in negative: 684, total: 3552


## Step 4: Applying Bayes' Theorem

For each keyword we compute:

$$P(\text{Positive} \mid \text{keyword}) = \frac{P(\text{keyword} \mid \text{Positive}) \times P(\text{Positive})}{P(\text{keyword})}$$

| Term | Formula | Meaning |
|------|---------|---------|
| **Prior** | P(Positive) | Baseline probability — 50% of all reviews are positive |
| **Likelihood** | P(keyword \| Positive) | How often the keyword appears in positive reviews |
| **Marginal** | P(keyword) | How often the keyword appears across all reviews |
| **Posterior** | P(Positive \| keyword) | Updated probability after seeing the keyword |


In [ ]:
def bayes_posterior(keyword: str):
    """
    Compute P(Positive | keyword) using Bayes' Theorem.

    Returns a dictionary with all intermediate values.
    """
    in_pos, in_neg, kw_total = count_keyword(keyword)

    if kw_total == 0:
        return None  # keyword not found in dataset

    prior      = num_pos / total            # P(Positive)
    likelihood = in_pos  / num_pos          # P(keyword | Positive)
    marginal   = kw_total / total           # P(keyword)
    posterior  = (likelihood * prior) / marginal  # P(Positive | keyword)

    return {
        "keyword"   : keyword,
        "in_pos"    : in_pos,
        "in_neg"    : in_neg,
        "kw_total"  : kw_total,
        "prior"     : prior,
        "likelihood": likelihood,
        "marginal"  : marginal,
        "posterior" : posterior,
    }

# Test with one keyword
result = bayes_posterior("excellent")
print("Prior      :", round(result["prior"],      4))
print("Likelihood :", round(result["likelihood"], 4))
print("Marginal   :", round(result["marginal"],   4))
print("Posterior  :", round(result["posterior"],  4))


## Step 5: Results for All Keywords


In [ ]:
results = [bayes_posterior(kw) for kw in ALL_KEYWORDS]

# ─── Print results table ──────────────────────────────────────
header = f"{'Keyword':<14} {'Prior':>8} {'Likelihood':>12} {'Marginal':>10} {'Posterior':>10} {'In +':>7} {'In -':>7} {'Total':>7}"
print(header)
print("─" * len(header))

for r in results:
    print(
        f"{r['keyword']:<14} "
        f"{r['prior']:>8.4f} "
        f"{r['likelihood']:>12.4f} "
        f"{r['marginal']:>10.4f} "
        f"{r['posterior']:>10.4f} "
        f"{r['in_pos']:>7} "
        f"{r['in_neg']:>7} "
        f"{r['kw_total']:>7}"
    )


## Step 6: Worked Example — "excellent"

Walking through the formula step by step with real numbers:


In [ ]:
r = bayes_posterior("excellent")

print("=" * 55)
print(f"  WORKED EXAMPLE — keyword: '{r['keyword']}' ")
print("=" * 55)
print(f"  Reviews in dataset        : {total:,}")
print(f"  Positive reviews          : {num_pos:,}")
print(f"  Reviews with keyword      : {r['kw_total']:,}")
print(f"  Keyword in positive       : {r['in_pos']:,}")
print()
print(f"  Prior      = {num_pos} / {total}   = {r['prior']:.4f}")
print(f"  Likelihood = {r['in_pos']} / {num_pos} = {r['likelihood']:.4f}")
print(f"  Marginal   = {r['kw_total']} / {total} = {r['marginal']:.4f}")
print()
print(f"  Posterior  = ({r['likelihood']:.4f} × {r['prior']:.4f}) / {r['marginal']:.4f}")
print(f"             = {r['posterior']:.4f}  →  {r['posterior']:.1%}")
print()
print(f"  Starting belief: 50.0%")
print(f"  After seeing '{r['keyword']}': {r['posterior']:.1%}")


## Step 7: Visualisation

A bar chart showing the posterior probability for each keyword, with the 50% prior marked as a reference line.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

keywords   = [r["keyword"]   for r in results]
posteriors = [r["posterior"] for r in results]
groups     = [kw in POSITIVE_KEYWORDS for kw in keywords]
colors     = ["#1B998B" if g else "#C44536" for g in groups]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(keywords, posteriors, color=colors, edgecolor="white", height=0.6)

# Prior reference line
ax.axvline(x=0.5, color="#555555", linestyle="--", linewidth=1.2, label="50% Prior (baseline)")

# Data labels
for bar, val in zip(bars, posteriors):
    ax.text(val + 0.01, bar.get_y() + bar.get_height() / 2,
            f"{val:.1%}", va="center", fontsize=11, color="#1E2761", fontweight="bold")

ax.set_xlabel("P(Positive | keyword)", fontsize=12)
ax.set_title("Bayesian Sentiment Analysis — Posterior Probabilities\nIMDB Dataset (50,000 reviews)", fontsize=14, fontweight="bold")
ax.set_xlim(0, 1.0)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))

pos_patch = mpatches.Patch(color="#1B998B", label="Positive keywords")
neg_patch = mpatches.Patch(color="#C44536", label="Negative keywords")
ax.legend(handles=[pos_patch, neg_patch, plt.Line2D([0], [0], color="#555555", linestyle="--")],
          labels=["Positive keywords", "Negative keywords", "50% Prior"],
          loc="lower right", fontsize=10)

ax.set_facecolor("#F8F9FC")
fig.patch.set_facecolor("white")
plt.tight_layout()
plt.savefig("bayesian_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved as bayesian_results.png")
